#### Feature Engineering in Machine Learning
    Feature engineering is the process of transforming raw data into meaningful, high‑quality features that help machine‑learning models learn patterns more effectively. It is one of the most influential steps in the ML pipeline and often contributes more to performance than the choice of algorithm.

#### Definition
    Feature engineering refers to the creation, transformation, encoding, and selection of features so that machine‑learning algorithms can interpret data more effectively. It includes cleaning data, encoding categorical variables, scaling numeric values, generating new features, and reducing dimensionality.

#### Usage
    Feature engineering is used to:

        - Convert raw data into model‑ready numerical representations

        - Capture non‑linear relationships that raw features cannot express

        - Improve the signal‑to‑noise ratio

        - Make data compatible with algorithms requiring specific formats

        - Incorporate domain knowledge into the dataset

#### Why It Matters
    Feature engineering is important because:

        - Models learn patterns more accurately with well‑structured features

        - It reduces noise and emphasizes meaningful structure

        - It improves generalization and reduces overfitting

        - It often leads to greater accuracy gains than switching algorithms

        - It helps models converge faster and more reliably

#### What It Includes
    1. Data Cleaning
        Handling missing values, fixing inconsistent formats, removing duplicates, and correcting outliers.

    2. Encoding Categorical Variables
        One‑hot encoding, ordinal encoding, target encoding, and frequency encoding.

    3. Scaling and Normalization
        Standardization, min–max scaling, and robust scaling.

    4. Transforming Numeric Features
        Log transforms, Box–Cox/Yeo–Johnson transforms, and binning continuous variables.

    5. Creating New Features
        Interaction terms, polynomial features, domain‑driven features, and aggregated statistics.

    6. Feature Selection and Reduction
        Correlation filtering, mutual information, Lasso, tree‑based importance, PCA, and autoencoders.


#### When to Use Feature Engineering
    Feature engineering is most useful when:

        - Raw data contains categorical variables

        - Numeric features are skewed, noisy, or on different scales
    
        - The dataset is small or medium‑sized

        - Domain knowledge suggests meaningful transformations

        - Model performance is low and needs additional structure

        - Using algorithms that require numeric, scaled, or clean inputs

#### Advantages
        - Higher accuracy

        - Better generalization

        - Improved interpretability

        - Faster training for scale‑sensitive models

        - Flexibility to incorporate domain expertise

        - Ability to handle messy real‑world data

#### Disadvantages and Limitations
        - Time‑consuming and iterative

        - Requires domain knowledge

        - Risk of overfitting with too many engineered features

        - Not always transferable across datasets

        - Can increase model complexity

        - Diminishing returns after a point


#### Conclusion
    Feature engineering is a foundational step in machine learning that transforms raw data into structured, meaningful features that models can learn from effectively. It enhances accuracy, stability, and interpretability, especially when data is messy or limited. Although it requires time, domain knowledge, and experimentation, strong feature engineering can turn an average model into a high‑performing one and is essential for building reliable, real‑world ML systems.


In [18]:
import pandas as pd          # Data manipulation
import numpy as np           # Numerical operations

import matplotlib.pyplot as plt   # Basic plotting
import seaborn as sns             # Statistical visualizations

import plotly.express as px       # Interactive plots (histograms, bar charts)

sns.set(style="whitegrid")


from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer


In [19]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
df_train_test = pd.read_csv("df_train_test.csv") 

df_train_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,capital_gain_bin,capital_loss_bin,net_capital,income_binary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train,1,0,2174,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train,0,0,0,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train,0,0,0,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train,0,0,0,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train,0,0,0,0


In [21]:
class FeatureCreator(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Net capital
        #X["net_capital"] = X["capital_gain"] - X["capital_loss"]

        # Interaction features
        X["age_hours_interaction"] = X["age"] * X["hours_per_week"]
        X["edu_hours_interaction"] = X["education_num"] * X["hours_per_week"]

        # Age groups
        X["age_group"] = pd.cut(
            X["age"],
            bins=[0, 25, 45, 65, 100],
            labels=["Young", "Mid", "Senior", "Elder"]
        )

        return X


In [22]:
creator = FeatureCreator()
df_train_test = creator.fit_transform(df_train_test)


#### Why you must split before encoding and scaling

You split back into train_df and test_df before encoding and scaling because encoders and scalers must learn only from the training data. This is one of the most important rules in machine learning to avoid data leakage.

##### Encoding and scaling are learning steps. They compute statistics from the data:

        - OneHotEncoder learns all categories present in the data.

        - StandardScaler learns mean and standard deviation.

        - Target encoders learn target averages.

        - PCA learns principal components.

        - If you fit these on the combined dataset (train + test), the model indirectly “sees” the test data during training.

    This contaminates the training process.

#### What goes wrong if you encode/scale before splitting

    1. Leakage of category information
        If the test set contains a category not present in the training set, and you fit OHE on combined data, the encoder will include that category.
        Your model now knows something about the test distribution.

    2. Leakage of distribution statistics
        If you scale on combined data:

        mean = (train + test)
    
        std = (train + test)

        Your model is trained on values normalized using test information.
        This artificially improves performance and makes evaluation invalid.

    3. Leakage in imputation
        If you compute median/mean on combined data, the test values influence the imputation strategy.

    4. Leakage in PCA or other transformations
        PCA learns directions of maximum variance.
        If test data is included, PCA is influenced by test variance.



In [23]:
#Split Back into Train and Test Using source

train_df = df_train_test[df_train_test["source"] == "train"].drop(columns=["source"])
test_df  = df_train_test[df_train_test["source"] == "test"].drop(columns=["source"])


In [48]:
#Define Columns for Encoding and Scaling

categorical_cols = [
    "workclass", "education", "marital_status", "occupation",
    "relationship", "race", "sex", "native_country", "age_group"
]

numeric_cols = [
    "age", "education_num", "hours_per_week",
    "capital_gain", "capital_loss", "net_capital",
    "age_hours_interaction", "edu_hours_interaction"
]

skewed_cols = ["capital_gain", "capital_loss", "net_capital"]

numeric_cols.append("net_capital")
skewed_cols.remove("net_capital")



In [49]:
#Add Imputers to the Preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_cols),

        ("log_skewed", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("log", FunctionTransformer(np.log1p)),
            ("scale", StandardScaler())
        ]), skewed_cols),

        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ],
    remainder="drop"
)


In [50]:
#Build Preprocessing Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        
        ("log_skewed", Pipeline([
            ("log", FunctionTransformer(np.log1p)),
            ("scale", StandardScaler())
        ]), skewed_cols),
        
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="drop"
)



In [51]:
# Fit Preprocessor ONLY on Train Data

X_train = train_df.drop(columns=["income", "income_binary"])
y_train = train_df["income_binary"]

X_test = test_df.drop(columns=["income", "income_binary"])
y_test = test_df["income_binary"]

preprocessor.fit(X_train)


ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'education_num', 'hours_per_week',
                                  'capital_gain', 'capital_loss', 'net_capital',
                                  'age_hours_interaction',
                                  'edu_hours_interaction', 'net_capital']),
                                ('log_skewed',
                                 Pipeline(steps=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>)),
                                                 ('scale', StandardScaler())]),
                                 ['capital_gain', 'capital_loss']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['workclass', 'education', 'marital_status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'native_country', 'age_group'])])

In [52]:
#Transform Train and Test

X_train_ready = preprocessor.transform(X_train)
X_test_ready  = preprocessor.transform(X_test)

print("Train shape:", X_train_ready.shape)
print("Test shape:", X_test_ready.shape)


Train shape: (32537, 117)
Test shape: (16276, 117)


#### The Adult dataset originally contains 48,842 rows (32,561 train + 16,281 test), but cleaned versions remove rows with missing values (`?`), duplicates, and formatting issues. After cleaning, the dataset typically reduces to around 48,813 rows. In this project, After removing, the final split is 32,537 training rows and 16,276 test rows. Train/test splitting is done before preprocessing to avoid data leakage. A ColumnTransformer then applies scaling to numeric features, log-scaling to skewed features, and OneHotEncoding to categorical features, producing 117 final model-ready features.
